Intelligent Road Safety Analytics: Accident Severity Prediction and Risk Zone Scoring using Big Data Pipeline

Data Loading and Cleaning using PySpark

### Objective
The objective of this notebook is to load the road accident dataset into PySpark, inspect its structure, assess data quality, identify missing or inconsistent values, and prepare a clean dataset for Spark SQL analysis and machine learning.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = (
    SparkSession.builder
    .appName("Intelligent Road Safety Analytics")
    .master("local[*]")
    .getOrCreate()
)

26/08/02 23:14:25 WARN Utils: Your hostname, LAPTOP-S89A4J3G resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/02 23:14:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/02 23:14:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Load the Dataset

In this step, the road accident dataset is loaded into a PySpark DataFrame. The dataset contains historical accident records that will be used for data cleaning, Spark SQL analysis, exploratory data analysis (EDA), and machine learning.

In [3]:
df = spark.read.csv(
    "../data/Road Accident Data.csv",
    header=True,
    inferSchema=True
)

In [4]:
print(f"Number of Rows    : {df.count()}")
print(f"Number of Columns : {len(df.columns)}")

Number of Rows    : 307973
Number of Columns : 23


In [5]:
df.show(5, truncate=False)

+--------------+-------------+-----+-----------+----+------------------------+-----------------------+-----------------+---------+---------------------+--------------------------+-------------------+---------+--------------------+------------------+-------------------+-----------------------+------------------+-----------+-------------------+-------------------+------------------+---------------------+
|Accident_Index|Accident Date|Month|Day_of_Week|Year|Junction_Control        |Junction_Detail        |Accident_Severity|Latitude |Light_Conditions     |Local_Authority_(District)|Carriageway_Hazards|Longitude|Number_of_Casualties|Number_of_Vehicles|Police_Force       |Road_Surface_Conditions|Road_Type         |Speed_limit|Time               |Urban_or_Rural_Area|Weather_Conditions|Vehicle_Type         |
+--------------+-------------+-----+-----------+----+------------------------+-----------------------+-----------------+---------+---------------------+--------------------------+---------

In [6]:
df.printSchema()

root
 |-- Accident_Index: string (nullable = true)
 |-- Accident Date: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day_of_Week: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Junction_Control: string (nullable = true)
 |-- Junction_Detail: string (nullable = true)
 |-- Accident_Severity: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Light_Conditions: string (nullable = true)
 |-- Local_Authority_(District): string (nullable = true)
 |-- Carriageway_Hazards: string (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Number_of_Casualties: integer (nullable = true)
 |-- Number_of_Vehicles: integer (nullable = true)
 |-- Police_Force: string (nullable = true)
 |-- Road_Surface_Conditions: string (nullable = true)
 |-- Road_Type: string (nullable = true)
 |-- Speed_limit: integer (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Urban_or_Rural_Area: string (nullable = true)
 |-- Weather_Conditions: st

In [7]:
df.select("Accident Date").show(20, truncate=False)

+-------------+
|Accident Date|
+-------------+
|01-01-2021   |
|01-05-2021   |
|01-04-2021   |
|01-05-2021   |
|01-06-2021   |
|01-01-2021   |
|01-08-2021   |
|01-02-2021   |
|01-07-2021   |
|01-10-2021   |
|01-07-2021   |
|1/16/2021    |
|01-12-2021   |
|01-09-2021   |
|1/17/2021    |
|1/25/2021    |
|1/26/2021    |
|1/26/2021    |
|1/19/2021    |
|1/27/2021    |
+-------------+
only showing top 20 rows



In [8]:
from pyspark.sql.functions import col, to_date, coalesce

df = df.withColumn(
    "Accident Date",
    coalesce(
        to_date(col("Accident Date"), "dd-MM-yyyy"),
        to_date(col("Accident Date"), "M/d/yyyy")
    )
)

In [9]:
df.printSchema()

root
 |-- Accident_Index: string (nullable = true)
 |-- Accident Date: date (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day_of_Week: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Junction_Control: string (nullable = true)
 |-- Junction_Detail: string (nullable = true)
 |-- Accident_Severity: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Light_Conditions: string (nullable = true)
 |-- Local_Authority_(District): string (nullable = true)
 |-- Carriageway_Hazards: string (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Number_of_Casualties: integer (nullable = true)
 |-- Number_of_Vehicles: integer (nullable = true)
 |-- Police_Force: string (nullable = true)
 |-- Road_Surface_Conditions: string (nullable = true)
 |-- Road_Type: string (nullable = true)
 |-- Speed_limit: integer (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Urban_or_Rural_Area: string (nullable = true)
 |-- Weather_Conditions: stri

In [10]:
from pyspark.sql.functions import col

df.filter(col("Accident Date").isNull()).count()

0

## Missing Value Analysis

Missing values can affect data quality and machine learning performance. This section identifies columns containing null values so that appropriate data cleaning techniques can be applied.

In [11]:
from pyspark.sql.functions import col, when, count

missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_values.show(truncate=False)

+--------------+-------------+-----+-----------+----+----------------+---------------+-----------------+--------+----------------+--------------------------+-------------------+---------+--------------------+------------------+------------+-----------------------+---------+-----------+----+-------------------+------------------+------------+
|Accident_Index|Accident Date|Month|Day_of_Week|Year|Junction_Control|Junction_Detail|Accident_Severity|Latitude|Light_Conditions|Local_Authority_(District)|Carriageway_Hazards|Longitude|Number_of_Casualties|Number_of_Vehicles|Police_Force|Road_Surface_Conditions|Road_Type|Speed_limit|Time|Urban_or_Rural_Area|Weather_Conditions|Vehicle_Type|
+--------------+-------------+-----+-----------+----+----------------+---------------+-----------------+--------+----------------+--------------------------+-------------------+---------+--------------------+------------------+------------+-----------------------+---------+-----------+----+-------------------+-

In [12]:
before_rows = df.count()

df = df.dropna()

after_rows = df.count()

print("Rows before cleaning :", before_rows)
print("Rows after cleaning  :", after_rows)
print("Rows removed         :", before_rows - after_rows)

Rows before cleaning : 307973
Rows after cleaning  : 300492
Rows removed         : 7481


In [13]:
missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_values.show()

+--------------+-------------+-----+-----------+----+----------------+---------------+-----------------+--------+----------------+--------------------------+-------------------+---------+--------------------+------------------+------------+-----------------------+---------+-----------+----+-------------------+------------------+------------+
|Accident_Index|Accident Date|Month|Day_of_Week|Year|Junction_Control|Junction_Detail|Accident_Severity|Latitude|Light_Conditions|Local_Authority_(District)|Carriageway_Hazards|Longitude|Number_of_Casualties|Number_of_Vehicles|Police_Force|Road_Surface_Conditions|Road_Type|Speed_limit|Time|Urban_or_Rural_Area|Weather_Conditions|Vehicle_Type|
+--------------+-------------+-----+-----------+----+----------------+---------------+-----------------+--------+----------------+--------------------------+-------------------+---------+--------------------+------------------+------------+-----------------------+---------+-----------+----+-------------------+-

In [14]:
before_rows = df.count()

after_drop_duplicates = df.dropDuplicates().count()

duplicate_records = before_rows - after_drop_duplicates

print("Total Duplicate Records:", duplicate_records)

Total Duplicate Records: 1


In [15]:
if duplicate_records > 0:
    df = df.dropDuplicates()
    print("Duplicate records removed.")
else:
    print("No duplicate records found.")

Duplicate records removed.


In [16]:
print("Total Rows:", df.count())

Total Rows: 300491


## Data Validation

Before analysis, the dataset is validated to identify invalid or inconsistent values that could affect the quality of insights and machine learning models.

In [17]:
df.select("Accident_Severity").distinct().show(truncate=False)

+-----------------+
|Accident_Severity|
+-----------------+
|Slight           |
|Fatal            |
|Serious          |
+-----------------+



In [18]:
df.groupBy("Speed_limit").count().orderBy("Speed_limit").show()

+-----------+------+
|Speed_limit| count|
+-----------+------+
|         10|     2|
|         15|     2|
|         20|  2788|
|         30|194511|
|         40| 25163|
|         50|  9999|
|         60| 46056|
|         70| 21970|
+-----------+------+



In [19]:
df.describe("Number_of_Casualties").show()

+-------+--------------------+
|summary|Number_of_Casualties|
+-------+--------------------+
|  count|              300491|
|   mean|  1.3594516973886073|
| stddev|  0.8182843932704761|
|    min|                   1|
|    max|                  48|
+-------+--------------------+



In [20]:
df.filter(col("Number_of_Casualties") < 0).count()

0

In [21]:
df.describe("Number_of_Vehicles").show()

+-------+------------------+
|summary|Number_of_Vehicles|
+-------+------------------+
|  count|            300491|
|   mean|1.8305240423174072|
| stddev| 0.711746774908921|
|    min|                 1|
|    max|                32|
+-------+------------------+



In [22]:
df.filter(col("Number_of_Vehicles") <= 0).count()

0

In [23]:
df.describe("Latitude", "Longitude").show()

+-------+------------------+-------------------+
|summary|          Latitude|          Longitude|
+-------+------------------+-------------------+
|  count|            300491|             300491|
|   mean|52.485284658702064|-1.3618635096858096|
| stddev|1.3413260023379485| 1.3540121408080485|
|    min|         49.914488|          -7.516225|
|    max|         60.598055|           1.759398|
+-------+------------------+-------------------+



In [24]:
df.write.mode("overwrite").option("header", True).csv("../cleaned_data/road_accident_cleaned")

In [25]:
df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("../cleaned_data/road_accident_cleaned")